<a href="https://colab.research.google.com/github/imabigger/my_DS_recipe_book/blob/main/%EB%AA%A8%EA%B8%B0%EB%B9%84%ED%96%89%EA%B2%BD%EB%A1%9C%EC%98%88%EC%B8%A1/%EC%B5%9C%EC%A2%85_%EB%AA%A8%EB%8D%B8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

전략 [3] 양방향 GRU + Self-Attention 시퀀스 백본 업그레이드
현재 단방향 single-layer LSTM 구조는 과거 9개 패치의 후반부 정보에 편향되거나 난류 노이즈와 본질적 기동 시그널을 혼동하기 쉽습니다. 시퀀스 인코더의 용량(Capacity)을 확장하기 위해 Bidirectional GRU 레이어 위에 시간축에 대한 Self-Attention Pooling을 결합하는 구조를 제안합니다.

과거 패치 간의 인과적 선후 관계를 양방향으로 추적하고, 최종 은닉 상태를 추출할 때 마지막 9번째 타임스텝만 단독 매핑하는 대신, 과거 9개 스텝 중 "기동 변화의 결정적 힌트가 고여 있는 타임스텝"에 동적으로 주목(Attention Weighting)하여 컨텍스트 벡터 $h_{\text{hist}}$를 형성하게 만듭니다.

In [ ]:
!cp "/content/drive/MyDrive/ai/모기비행궤적예측ai경진대회/open.zip" "/content/"
!unzip -q "/content/open.zip" -d "/content/"

In [ ]:
import os
import glob
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset
from sklearn.model_selection import KFold
from tqdm import tqdm



class MosquitoGeometryFeatureExtractor:
    def __init__(self):
        self.dt = 0.04
        self.eps = 1e-8

    def __call__(self, x_raw):
        features_seq_T = []
        features_seq_NB = []
        historical_seq = []

        T_list, N_list, B_list = [], [], []
        v_list = []
        v_norms = []
        kappas = []

        # 1차 루프: 기본 프레네 기저 벡터 및 물리 속도 벡터 계산
        for i in range(2, 11):
            p0, p_m1, p_m2 = x_raw[i], x_raw[i-1], x_raw[i-2]
            v = (p0 - p_m1) / self.dt
            a = (p0 - 2 * p_m1 + p_m2) / (self.dt ** 2)

            v_norm = np.linalg.norm(v) + self.eps
            cross_va = np.cross(v, a)
            cross_norm = np.linalg.norm(cross_va) + self.eps

            T_vec = v / v_norm
            if cross_norm > self.eps:
                kappa = cross_norm / (v_norm ** 3)
                B_vec = cross_va / cross_norm
                N_vec = np.cross(B_vec, T_vec)
            else:
                kappa = 0.0
                B_vec = np.array([0.0, 0.0, 0.0])
                N_vec = np.array([0.0, 0.0, 0.0])

            T_list.append(T_vec)
            N_list.append(N_vec)
            B_list.append(B_vec)
            v_list.append(v)
            v_norms.append(v_norm)
            kappas.append(np.clip(kappa, 0.0, 1000.0))

        # 최종 시점(10번째 패치)의 프레네 프레임 R_10 확정
        R_current = np.column_stack([T_list[-1], N_list[-1], B_list[-1]])

        # 2차 루프: 정렬된 피처(v_t, v_n, v_b) 및 프레임 전이(vec_M) 계산
        for idx in range(9):
            i = idx + 2
            kappa = kappas[idx]
            log_kappa = np.log(kappa + self.eps)
            v_norm = v_norms[idx]

            # 현재 시점의 프레임 R_i
            R_curr = np.column_stack([T_list[idx], N_list[idx], B_list[idx]])

            # torsion 및 전이 행렬 M 계산
            if idx > 0:
                dt_disp = v_norm * self.dt
                torsion = np.dot(B_list[idx-1] - B_list[idx], N_list[idx]) / (dt_disp + self.eps)
                R_prev = np.column_stack([T_list[idx-1], N_list[idx-1], B_list[idx-1]])
                M_matrix = np.dot(R_prev.T, R_curr)
            else:
                torsion = 0.0
                M_matrix = np.eye(3)

            vec_M = M_matrix.flatten()

            # 정렬된 국소 속도 계산 (v_aligned = [v_t, v_n, v_b])
            v_aligned = np.dot(R_current.T, v_list[idx])
            v_t = v_aligned[0]
            v_n = v_aligned[1]
            v_b = v_aligned[2]

            # 과거 target_local 변위 계산
            if i + 2 <= 10:
                delta_local = np.dot(R_curr.T, x_raw[i+2] - x_raw[i])
            else:
                delta_local = np.array([np.nan, np.nan, np.nan])
            historical_seq.append(delta_local)

            # [T 예측용 피처 조립] v_norm, v_t, log_kappa, T_x, T_y, T_z (총 6차원)
            feat_T = [v_norm, v_t, log_kappa] + T_list[idx].tolist()
            features_seq_T.append(feat_T)

            # [NB 예측용 피처 조립] log_kappa, torsion, vec_M, v_n, v_b, v_norm (총 14차원)
            feat_NB = [log_kappa, torsion] + vec_M.tolist() + [v_n, v_b, v_norm]
            features_seq_NB.append(feat_NB)

        return (
            torch.tensor(features_seq_T, dtype=torch.float32),
            torch.tensor(features_seq_NB, dtype=torch.float32),
            torch.tensor(historical_seq, dtype=torch.float32),
            R_current
        )

In [ ]:
class MosquitoFrenetDataset(Dataset):
    def __init__(self, data_dir, label_path):
        self.file_paths = sorted(glob.glob(os.path.join(data_dir, "*.csv")))
        self.labels_df = pd.read_csv(label_path)

        if 'id' in self.labels_df.columns:
            self.labels_df.set_index('id', inplace=True)

        self.extractor = MosquitoGeometryFeatureExtractor()
        self.precomputed_data = []

        print("T 및 NB 피처 전용 데이터셋 분리 구축 중...")
        for file_path in tqdm(self.file_paths):
            df = pd.read_csv(file_path)
            x_raw = df[['x', 'y', 'z']].values[:11, :]
            file_name = os.path.basename(file_path).split('.')[0]

            features_T, features_NB, historical_tensor, R_current = self.extractor(x_raw)

            y_abs = self.labels_df.loc[file_name, ['x', 'y', 'z']].values.astype(np.float32)
            p0_final = x_raw[-1]

            delta_global = y_abs - p0_final
            target_local = np.dot(R_current.T, delta_global)

            self.precomputed_data.append({
                'features_T': features_T,                       #
                'features_NB': features_NB,                     #
                'historical_targets': historical_tensor,
                'target_local': torch.tensor(target_local, dtype=torch.float32),
                'target_global': torch.tensor(delta_global, dtype=torch.float32),
                'R_matrix': torch.from_numpy(R_current).float()
            })

    def __len__(self):
        return len(self.precomputed_data)

    def __getitem__(self, idx):
        return self.precomputed_data[idx]

In [ ]:
class MosquitoPureRegressionCascadeModel(nn.Module):
    def __init__(self, input_dim_t=6, input_dim_nb=14, hidden_dim=64):
        super().__init__()
        self.lstm_t = nn.LSTM(input_dim_t, hidden_dim, batch_first=True, num_layers=8)
        self.lstm_nb = nn.LSTM(input_dim_nb, hidden_dim, batch_first=True, num_layers=8)

        # 1단계: T 방향의 연속 실수를 직접 예측하는 다층 퍼셉트론 헤드
        self.t_head = nn.Sequential(
            nn.Linear(hidden_dim, 32),
            nn.ReLU(),
            nn.Linear(32, 1)  # Output: 단일 실수 값 [pred_t]
        )

        # 2단계: N, B 방향 조건부 예측 헤드
        self.nb_head = nn.Sequential(
            nn.Linear(hidden_dim + 1, 32),
            nn.ReLU(),
            nn.Linear(32, 2)  # Output: [pred_n, pred_b]
        )

    def forward(self, x_t, x_nb, r_matrix=None):
        # 1단계 분기 연산 (T 시스템)
        lstm_t_out, _ = self.lstm_t(x_t)
        h_t = lstm_t_out[:, -1, :]
        pred_t = self.t_head(h_t)  # (Batch, 1) 연속 변위 직접 추정

        # 2단계 분기 연산 (NB 시스템)
        lstm_nb_out, _ = self.lstm_nb(x_nb)
        h_nb = lstm_nb_out[:, -1, :]

        # 그래디언트 충돌 방지를 위한 진행 축 정보 분리
        pred_t_detached = pred_t.detach()

        nb_input = torch.cat([h_nb, pred_t_detached], dim=-1)
        pred_nb = self.nb_head(nb_input)
        pred_n = pred_nb[:, 0:1]
        pred_b = pred_nb[:, 1:2]

        pred_local = torch.cat([pred_t, pred_n, pred_b], dim=-1)

        # 전역 공간 3차원 복원
        if r_matrix is not None:
            pred_local_tmp = pred_local.unsqueeze(-1)
            pred_global = torch.bmm(r_matrix, pred_local_tmp).squeeze(-1)
            return pred_local, pred_global

        return pred_local, None

In [ ]:
# 시간 축 정보 압착을 위한 Self-Attention Pooling 모듈 정의
class SelfAttentionPooling(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.W = nn.Linear(input_dim, input_dim)
        self.v = nn.Linear(input_dim, 1, bias=False)

    def forward(self, seq_outputs):
        # seq_outputs 형태: (Batch, Seq_Len, input_dim)
        scores = self.v(torch.tanh(self.W(seq_outputs)))  # (Batch, Seq_Len, 1)
        weights = F.softmax(scores, dim=1)  # 시간 축 내적 정규화
        context = torch.sum(weights * seq_outputs, dim=1)  # 가중합 컨텍스트 도출 (Batch, input_dim)
        return context

# 전략 3번: 양방향 GRU + Self-Attention Pooling 결합 신경망 구조
class MosquitoAttentionBiGRUCascadeModel(nn.Module):
    def __init__(self, input_dim_t=6, input_dim_nb=14, hidden_dim=64):
        super().__init__()
        # 양방향 제어를 위해 bidirectional=True 활성화
        self.gru_t = nn.GRU(input_dim_t, hidden_dim, batch_first=True, num_layers=1, bidirectional=True)
        self.gru_nb = nn.GRU(input_dim_nb, hidden_dim, batch_first=True, num_layers=1, bidirectional=True)

        # 양방향 출력 차원은 2 * hidden_dim 이 되므로 풀링 입력 차원 매핑
        self.attn_pooling_t = SelfAttentionPooling(2 * hidden_dim)
        self.attn_pooling_nb = SelfAttentionPooling(2 * hidden_dim)

        self.t_head = nn.Sequential(
            nn.Linear(2 * hidden_dim, 32),
            nn.ReLU(),
            nn.Linear(32, 1)
        )

        # NB 헤드는 T 예측 성분(1차원)이 카스케이드 결합되므로 입력 차원에 1을 더함
        self.nb_head = nn.Sequential(
            nn.Linear(2 * hidden_dim + 1, 32),
            nn.ReLU(),
            nn.Linear(32, 2)
        )

    def forward(self, x_t, x_nb, r_matrix=None):
        # 1단계: T 시스템 연산
        gru_t_out, _ = self.gru_t(x_t)  # (Batch, 9, 2 * hidden_dim)
        h_t_attn = self.attn_pooling_t(gru_t_out)  # 시간 분산 압착 완료 (Batch, 2 * hidden_dim)
        pred_t = self.t_head(h_t_attn)

        # 2단계: NB 시스템 연산
        gru_nb_out, _ = self.gru_nb(x_nb)  # (Batch, 9, 2 * hidden_dim)
        h_nb_attn = self.attn_pooling_nb(gru_nb_out)  # 시간 분산 압착 완료 (Batch, 2 * hidden_dim)

        # 수치 최적화 평면 분리를 위한 그래디언트 디태치
        pred_t_detached = pred_t.detach()

        nb_input = torch.cat([h_nb_attn, pred_t_detached], dim=-1)  # (Batch, 2 * hidden_dim + 1)
        pred_nb = self.nb_head(nb_input)
        pred_n = pred_nb[:, 0:1]
        pred_b = pred_nb[:, 1:2]

        pred_local = torch.cat([pred_t, pred_n, pred_b], dim=-1)

        if r_matrix is not None:
            pred_local_tmp = pred_local.unsqueeze(-1)
            pred_global = torch.bmm(r_matrix, pred_local_tmp).squeeze(-1)
            return pred_local, pred_global

        return pred_local, None

In [ ]:
# [4] 순수 회귀용 분리형 손실 함수
class PureRegressionTNBLoss(nn.Module):
    def __init__(self):
        super().__init__()
        # T의 아웃라이어 분포 방어를 위해 Huber 손실 함수(Smooth L1) 채택
        self.huber_loss = nn.SmoothL1Loss(beta=0.02)
        self.mse_loss = nn.MSELoss()

    def forward(self, pred_local, target_local, stage=1):
        t_gt = target_local[:, 0:1]
        nb_gt = target_local[:, 1:3]

        pred_t = pred_local[:, 0:1]
        pred_nb = pred_local[:, 1:3]

        if stage == 1:
            loss_t = self.huber_loss(pred_t, t_gt)
            return loss_t
        else:
            loss_nb = self.mse_loss(pred_nb, nb_gt)
            return loss_nb


class AdvancedGlobalHingeLoss(nn.Module):
    def __init__(self, beta=1.0, gamma=10.0):
        super().__init__()
        self.beta = beta        # 전역 L2 최적화 가중치
        self.gamma = gamma      # 1cm 한계선 돌파용 강력한 힌지 페널티 가중치
        self.huber_loss = nn.SmoothL1Loss(beta=0.02)

    def forward(self, pred_local=None, target_local=None, pred_global=None, target_global=None, stage=2):
        if stage == 1:
            # Stage 1: 로컬 프레네 좌표계 상에서의 T 변위 직접 회귀 (Huber Loss)
            # pred_local[:, 0:1] 형태: (Batch, 1) -> 정답 ΔT 추종
            return self.huber_loss(pred_local[:, 0:1], target_local[:, 0:1])

        # Stage 2: 전역 xyz 공간 상에서의 유클리드 거리 직접 연산
        # pred_global 및 target_global 형태: (Batch, 3)
        error_vectors = target_global - pred_global
        l2_distances = torch.linalg.norm(error_vectors, dim=1) # (Batch,)

        # 1cm(0.01m) 초과 오차에 대한 Hinge Penalty 계산
        hinge_penalty = torch.clamp(l2_distances - 0.01, min=0.0)

        # 최종 평가지표 맞춤형 멀티태스크 손실 함수 산출
        loss_l2 = torch.mean(l2_distances ** 2)
        loss_hinge = torch.mean(hinge_penalty)

        total_loss = self.beta * loss_l2 + self.gamma * loss_hinge
        return total_loss

class PureLocalGeometryLoss(nn.Module):
    def __init__(self):
        super().__init__()
        self.huber_t = nn.SmoothL1Loss(beta=0.02)
        self.huber_nb = nn.SmoothL1Loss(beta=0.01) # NB의 미세 요동을 위한 국소 Huber 선언

    def forward(self, pred_local, target_local, stage=1):
        # target_local 형태: (Batch, 3) -> [T_gt, N_gt, B_gt]
        if stage == 1:
            # Stage 1: 국소 T 변위 직접 최적화
            return self.huber_t(pred_local[:, 0:1], target_local[:, 0:1])
        else:
            # Stage 2: 의도하신 대로 전역 변환 없이 오직 국소 N, B 변위 자체의 오차만 최소화
            return self.huber_nb(pred_local[:, 1:3], target_local[:, 1:3])

In [ ]:
def train_and_validate_flexible_pipeline(
    full_dataset,
    model_class,
    model_kwargs=None,
    nb_keywords=None,
    k_splits=5,
    stage1_epochs=200,
    stage2_epochs=200,
    patience=30,
    batch_size=256
):
    kf = KFold(n_splits=k_splits, shuffle=True, random_state=42)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    fold_r_hit_scores = []

    # 모델 생성을 위한 인자 및 동결 키워드 기본값 정의
    model_kwargs = model_kwargs if model_kwargs is not None else {}
    # 예: PureRegression 모델이면 ['lstm_nb', 'nb_head'], BiGRU 모델이면 ['gru_nb', 'nb_head', 'attn_pooling_nb']
    nb_keywords = nb_keywords if nb_keywords is not None else ['nb', 'NB']

    print(f"\n==================== {k_splits}-Fold 유연한 순차 교차 검증 파이프라인 ====================")
    for fold, (train_idx, val_idx) in enumerate(kf.split(full_dataset)):
        print(f"\n--- [Fold {fold + 1} / {k_splits}] 아키텍처 최적화 정렬 ---")

        train_sub = Subset(full_dataset, train_idx)
        val_sub = Subset(full_dataset, val_idx)

        train_dataloader = DataLoader(train_sub, batch_size=batch_size, shuffle=True, pin_memory=True)
        val_dataloader = DataLoader(val_sub, batch_size=batch_size, shuffle=False, pin_memory=True)

        # 1. 매개변수로 전달받은 아키텍처 클래스로부터 동적 모델 인스턴스화
        model = model_class(**model_kwargs).to(device)
        #criterion = AdvancedGlobalHingeLoss(beta=1.0, gamma=20.0)
        criterion = PureLocalGeometryLoss() # 순수 국소 오차 함수로 교체

        # ----------------------------------------------------
        # STAGE 1: T 시스템 회귀 모델 최적화
        # ----------------------------------------------------
        print(f"[Stage 1] {model_class.__name__} T-System 연속 회귀 학습 시작")

        # 키워드 리스트를 기반으로 NB 계열 파라미터를 제외한 T 파라미터만 필터링
        t_params = [
            p for name, p in model.named_parameters()
            if not any(kw in name for kw in nb_keywords)
        ]
        optimizer_t = torch.optim.Adam(t_params, lr=0.001)

        best_val_loss_t = float('inf')
        patience_counter = 0
        best_t_model_state = None

        for epoch in range(stage1_epochs):
            model.train()
            train_loss_t = 0.0
            for batch in train_dataloader:
                features_t = batch['features_T'].to(device)
                features_nb = batch['features_NB'].to(device)
                target_local = batch['target_local'].to(device)

                optimizer_t.zero_grad()
                pred_local, _ = model(features_t, features_nb)

                # 인자 매핑 명시화 수정
                loss = criterion(pred_local=pred_local, target_local=target_local, stage=1)
                loss.backward()
                optimizer_t.step()

                train_loss_t += loss.item() * features_t.size(0)

            model.eval()
            val_loss_t = 0.0
            with torch.no_grad():
                for batch in val_dataloader:
                    features_t = batch['features_T'].to(device)
                    features_nb = batch['features_NB'].to(device)
                    target_local = batch['target_local'].to(device)

                    pred_local, _ = model(features_t, features_nb)
                    loss = criterion(pred_local=pred_local, target_local=target_local, stage=1)
                    val_loss_t += loss.item() * features_t.size(0)

            epoch_train_loss = train_loss_t / len(train_sub)
            epoch_val_loss = val_loss_t / len(val_sub)

            if epoch_val_loss < best_val_loss_t:
                best_val_loss_t = epoch_val_loss
                patience_counter = 0
                best_t_model_state = model.state_dict()
            else:
                patience_counter += 1

            if (epoch + 1) % 10 == 0 or epoch == 0 or patience_counter >= patience:
                print(f"Stage 1 - Epoch [{epoch+1:03d}/{stage1_epochs}] | Train Huber T: {epoch_train_loss:.6f} | Val Huber T: {epoch_val_loss:.6f}")

            if patience_counter >= patience:
                break

        if best_t_model_state is not None:
            model.load_state_dict(best_t_model_state)

        # ----------------------------------------------------
        # STAGE 2: T 가중치 전역 동결 후 NB 최적화
        # ----------------------------------------------------
        print(f"\n[Stage 2] T 모듈 동결 및 NB-System 전역 힌지 학습 개시")

        # 키워드 매칭을 이용한 동적 파라미터 락(Lock) 제어
        for name, param in model.named_parameters():
            if any(kw in name for kw in nb_keywords):
                param.requires_grad = True
            else:
                param.requires_grad = False

        nb_params = [p for name, p in model.named_parameters() if p.requires_grad]
        optimizer_nb = torch.optim.Adam(nb_params, lr=0.001)

        best_val_loss_nb = float('inf')
        best_r_hit = 0.0
        patience_counter = 0

        for epoch in range(stage2_epochs):
            model.train()
            train_loss_nb = 0.0
            for batch in train_dataloader:
                features_t = batch['features_T'].to(device)
                features_nb = batch['features_NB'].to(device)
                target_local = batch['target_local'].to(device)

                optimizer_nb.zero_grad()
                # 훈련 단계에서는 전역 변환(r_matrix)을 수행하지 않고 오직 국소 변위만 출력
                pred_local, _ = model(features_t, features_nb)
                loss = criterion(pred_local=pred_local, target_local=target_local, stage=2)
                loss.backward()
                optimizer_nb.step()

                train_loss_nb += loss.item() * features_t.size(0)

            model.eval()
            val_loss_nb = 0.0
            val_hits = 0
            val_total = 0

            with torch.no_grad():
                for batch in val_dataloader:
                    features_t = batch['features_T'].to(device)
                    features_nb = batch['features_NB'].to(device)
                    target_local = batch['target_local'].to(device)
                    target_global = batch['target_global'].to(device)
                    r_matrix = batch['R_matrix'].to(device)

                    # 평가지표 산출을 위한 전역 xyz 복원은 오직 평가 검증(eval) 단계에서만 수행
                    pred_local, pred_global = model(features_t, features_nb, r_matrix)
                    loss = criterion(pred_local=pred_local, target_local=target_local, stage=2)
                    val_loss_nb += loss.item() * features_t.size(0)

                    # 1cm 이내 r-hit 스코어 계산
                    distances = torch.linalg.norm(pred_global - target_global, dim=1)
                    hits = (distances <= 0.01).float().sum().item()
                    val_hits += hits
                    val_total += features_t.size(0)

            epoch_train_loss_nb = train_loss_nb / len(train_sub)
            epoch_val_loss_nb = val_loss_nb / len(val_sub)
            epoch_r_hit = val_hits / val_total

            if epoch_val_loss_nb < best_val_loss_nb:
                best_val_loss_nb = epoch_val_loss_nb
                best_r_hit = epoch_r_hit
                patience_counter = 0
            else:
                patience_counter += 1

            if (epoch + 1) % 10 == 0 or epoch == 0 or patience_counter >= patience:
                print(f"Stage 2 - Epoch [{epoch+1:03d}/{stage2_epochs}] | Train Hinge: {epoch_train_loss_nb:.6f} | Val Hinge: {epoch_val_loss_nb:.6f} | Val r-hit: {epoch_r_hit * 100:.2f}%")

            if patience_counter >= patience:
                break

        print(f"-> Fold {fold + 1} 완료 | 최종 1cm r-hit 지표: {best_r_hit * 100:.2f}%")
        fold_r_hit_scores.append(best_r_hit)

    mean_r_hit = np.mean(fold_r_hit_scores)
    std_r_hit = np.std(fold_r_hit_scores)
    print(f"\n==================== {model_class.__name__} 최종 리포트 ====================")
    for i, score in enumerate(fold_r_hit_scores):
        print(f"Fold {i+1} r-hit: {score * 100:.2f}%")
    print(f"\n평균 r-hit 성능: {mean_r_hit * 100:.2f}% (± {std_r_hit * 100:.2f}%)")

    return fold_r_hit_scores

In [ ]:
train_data_dir = "/content/train"
train_label_path = "/content/train_labels.csv"
full_dataset = MosquitoFrenetDataset(train_data_dir, train_label_path)

T 및 NB 피처 전용 데이터셋 분리 구축 중...


  0%|          | 0/10000 [00:00<?, ?it/s]/tmp/ipykernel_6852/3721598151.py:105: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:253.)
  torch.tensor(historical_seq, dtype=torch.float32),
100%|██████████| 10000/10000 [00:32<00:00, 306.91it/s]


In [ ]:
pure_regression_scores = train_and_validate_flexible_pipeline(
    full_dataset=full_dataset,
    model_class=MosquitoPureRegressionCascadeModel,
    model_kwargs={'input_dim_t': 6, 'input_dim_nb': 14, 'hidden_dim': 64},
    nb_keywords=['lstm_nb', 'nb_head'], # 해당 레이어들만 Stage 2에서 활성화
    k_splits=5
)


==================== 5-Fold 유연한 순차 교차 검증 파이프라인 ====================

--- [Fold 1 / 5] 아키텍처 최적화 정렬 ---
[Stage 1] MosquitoPureRegressionCascadeModel T-System 연속 회귀 학습 시작
Stage 1 - Epoch [001/200] | Train Huber T: 0.036537 | Val Huber T: 0.014985
Stage 1 - Epoch [010/200] | Train Huber T: 0.002939 | Val Huber T: 0.003091
Stage 1 - Epoch [020/200] | Train Huber T: 0.002481 | Val Huber T: 0.002677
Stage 1 - Epoch [030/200] | Train Huber T: 0.002406 | Val Huber T: 0.002655
Stage 1 - Epoch [040/200] | Train Huber T: 0.002371 | Val Huber T: 0.002532
Stage 1 - Epoch [050/200] | Train Huber T: 0.002320 | Val Huber T: 0.002628
Stage 1 - Epoch [060/200] | Train Huber T: 0.002235 | Val Huber T: 0.002541
Stage 1 - Epoch [070/200] | Train Huber T: 0.002208 | Val Huber T: 0.002557
Stage 1 - Epoch [080/200] | Train Huber T: 0.002081 | Val Huber T: 0.002630
Stage 1 - Epoch [090/200] | Train Huber T: 0.001951 | Val Huber T: 0.002501
Stage 1 - Epoch [100/200] | Train Huber T: 0.001820 | Val Huber T: 0.00

In [ ]:
bigru_attention_scores = train_and_validate_flexible_pipeline(
    full_dataset=full_dataset,
    model_class=MosquitoAttentionBiGRUCascadeModel,
    model_kwargs={'input_dim_t': 6, 'input_dim_nb': 14, 'hidden_dim': 64},
    nb_keywords=['gru_nb', 'nb_head', 'attn_pooling_nb'], # Attention 컴포넌트 포함 동결 해제
    k_splits=5
)


==================== 5-Fold 유연한 순차 교차 검증 파이프라인 ====================

--- [Fold 1 / 5] 아키텍처 최적화 정렬 ---
[Stage 1] MosquitoAttentionBiGRUCascadeModel T-System 연속 회귀 학습 시작
Stage 1 - Epoch [001/200] | Train Huber T: 0.024539 | Val Huber T: 0.007683
Stage 1 - Epoch [010/200] | Train Huber T: 0.002748 | Val Huber T: 0.002952
Stage 1 - Epoch [020/200] | Train Huber T: 0.002631 | Val Huber T: 0.002834
Stage 1 - Epoch [030/200] | Train Huber T: 0.002542 | Val Huber T: 0.002689
Stage 1 - Epoch [040/200] | Train Huber T: 0.002433 | Val Huber T: 0.002668
Stage 1 - Epoch [050/200] | Train Huber T: 0.002380 | Val Huber T: 0.002624
Stage 1 - Epoch [060/200] | Train Huber T: 0.002337 | Val Huber T: 0.002679
Stage 1 - Epoch [070/200] | Train Huber T: 0.002365 | Val Huber T: 0.002648
Stage 1 - Epoch [080/200] | Train Huber T: 0.002234 | Val Huber T: 0.002548
Stage 1 - Epoch [090/200] | Train Huber T: 0.002278 | Val Huber T: 0.002644
Stage 1 - Epoch [100/200] | Train Huber T: 0.002160 | Val Huber T: 0.00